#### Baseline модель

Цель: получить честную точку отсчёта для моделей на исходных признаках.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
data_dir = Path('../data/raw')

df_train = pd.read_csv(data_dir/'train.csv')
df_test = pd.read_csv(data_dir/'test.csv')

#### Данные для baseline

Не используем `PassengerId`, `Name`, `Ticket` и `Cabin`. `Pclass` храним как категориальный признак: классы `1`, `2` и `3` не прерывные числа.

In [3]:
# Целевая переменная
target = 'Survived'

# Числовые признаки, их заполняем и масштабируем
numeric_feature = [
    'Age',
    'SibSp',
    'Parch',
    'Fare'
]

# Категориальные признаки, их заполняем и кодируем.
# Pclass считаем категорией, а не непрерывным числом.
categorical_features = [
    'Pclass',
    'Sex',
    'Embarked'
]

# Полный список признаков baseline модели
feature_columns = numeric_feature + categorical_features

X = df_train[feature_columns]
y = df_train[target]

X.head()

,Age,SibSp,Parch,Fare,Pclass,Sex,Embarked
0,22.0,1,0,7.2500,3,male,S
1,38.0,1,0,71.2833,1,female,C
2,26.0,0,0,7.9250,3,female,S
3,35.0,1,0,53.1000,1,female,S
4,35.0,0,0,8.0500,3,male,S


#### Pipeline

Все преобразования выполняются внутри пайплайна. На каждом CV-фолде импутер, скейлер и энкодер обучаются только на train части фолда. Validation часть остается невидимой до оценки модели.

In [4]:
# Для числовых признаков:
# 1. Заполняем пропуски медианой train-части фолда.
# 2. Масштабируем значения для Logistic Regression.
numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

# AgeMissing создаем отдельной веткой. Так индикатор пропуска
# остается бинарным: 0 возраст известен, 1 возраст пропущен (мы
# не применяем к нему StandardScaler).
age_missing_pipeline = MissingIndicator(features='all')

# Категориальные признаки:
# 1. Заполняем пропуски самым частым значением.
# 2. OneHotEncoder преобразует категории в 0/1 столбцы.
# handle_unknown='ignore' защищает от неизвестной категории в test.
categorical_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# Применяем обработку к каждому набору признаков.
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_feature),
        ('age_missing', age_missing_pipeline, ['Age']),
        ('categorical', categorical_pipeline, categorical_features)
    ]
)

model = LogisticRegression(max_iter=300)

# Создаем единый пайплайн
baseline_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ]
)

#### Cross validation

Используем пять stratifield-фолдов: в каждом сохраняется примерно одинаковая доля выживших и невыживших пассажиров.

In [5]:
# Делим train на 5 воспроизводимых стратифицированных фолдов.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# На каждом фолде пайплайн обучается на 80% данных
# и оценивается по accuracy на оставшихся 20%.
cv_scores = cross_validate(
    baseline_pipeline,
    X,
    y,
    cv=cv,
    scoring='accuracy'
)

# Сохраняем accuracy каждого фолда в таблицу
fold_scores = cv_scores['test_score']

baseline_result = pd.DataFrame(
    {
        'Фолд': range(1, 6),
        'Accuracy': fold_scores
    }
)

display(
    baseline_result.style
    .hide(axis='index')
    .format({'Accuracy': '{:.2%}'})
)

# Среднее — оценка качества baseline.
# Стандартное отколнение — разброс качества между фолдами.
print(f'Средний CV accuracy: {fold_scores.mean():.2%}')
print(f'Стандартное отклонение: {fold_scores.std():.2%}')

Фолд,Accuracy
1,76.54%
2,80.34%
3,79.78%
4,78.65%
5,82.02%


Средний CV accuracy: 79.46%
Стандартное отклонение: 1.82%
